# PTA Data Analysis: Jupyter Notebook Lab
## Radio Astronomy — University of Manitoba · Lecture 2

**Tools:** `hasasia` · `holodeck` · `numpy` · `matplotlib`
**Data:** NANOGrav 15-yr public data release — Agazie et al. 2023, ApJL 951 L8/L9

> **Instructions:** Run every *code* cell in order. After each section, fill in
> the *answer cells* with your responses to the worksheet questions.
> Refer to your Lecture 2 student handout as you go.

---

## Before opening this notebook

Run the two setup scripts **once** in a terminal, then launch Jupyter:

```bash
# 1. Create the Python virtual environment and install all packages
bash setup.sh

# 2. Download the NANOGrav 15-yr dataset and write pta_paths.sh
bash get_data.sh

# 3. Activate the environment and start Jupyter
source pta_env/bin/activate
jupyter lab PTA_Lab_Student.ipynb
```

### Checklist — confirm before running Cell 0

- [ ] `setup.sh` completed without errors
- [ ] The active Jupyter kernel is **Python 3 (pta_env)**
      *(top-right corner of Jupyter Lab)*
- [ ] Either `Student_data/ng15yr_light_data.pkl` is present (handed out with
      the lab), **or** `get_data.sh` has run and `pta_paths.sh` exists

> **If you were given the light data file**, you do not need the 60 GB
> NANOGrav tarball at all — Cells 1 and 2 detect the file and skip the build.
>
> **Custom data location:** `bash get_data.sh /path/to/data`
> — `pta_paths.sh` is rewritten automatically.

In [ ]:
# ── Cell 0: Environment check + imports ─────────────────────────────────
import sys, os, glob, json, pickle, logging, warnings
from itertools import combinations
from scipy.stats import binned_statistic
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── PTA-specific imports ─────────────────────────────────────────────────
import hasasia
import hasasia.sensitivity as hsen
import hasasia.sim as hsim

import holodeck as holo
from holodeck import sams, hardening, host_relations
holo.log.setLevel(logging.WARNING)   # holodeck logs at INFO by default

fyr      = 1.0 / (365.25 * 24 * 3600)      # 1/yr in Hz
BUNDLE   = 'Student_data/ng15yr_light_data.pkl'
CACHEDIR = 'Student_data/_cache'

print('✓ All imports successful.')
print('  hasasia version  :', hasasia.__version__)
print('  holodeck version :', holo.__version__)
print()
print('Ready. Run cells in order: Cell 1 locates the data, Cell 2 builds the')
print('light data bundle (only if it is missing), Cell 3 loads it.')

---
## Section 0 · Setup and Data Loading
*(Worksheet: Section 1 — 'Setup and Data Loading')*

In [ ]:
# ── Cell 1: Locate the data ─────────────────────────────────────────────
# If the light data bundle is already here, there is nothing to locate: the
# bundle holds everything the rest of the lab needs (~0.3 MB instead of 60 GB).

def load_pta_paths(sh_file='pta_paths.sh'):
    """Read NG15_* assignments from pta_paths.sh in pure Python."""
    paths = {}
    with open(sh_file) as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, _, val = line.partition('=')
            val = val.strip().strip('"').strip("'")
            if key.strip().startswith('NG15_'):
                paths[key.strip()] = val
    return paths

# The release ships single-telescope subsets of some pulsars, named <PSR>ao
# and <PSR>gbt next to the combined <PSR>.  They are not separate pulsars:
# keeping them would pair a pulsar with part of its own data at zero angular
# separation, where the Hellings-Downs coefficient is undefined.
SKIP_SUFFIXES = ('.NoRedNoise', '.predictive', '.10yr')
TEL_SUFFIXES  = ('ao', 'gbt')

def is_canonical(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return not any(stem.endswith(s) for s in SKIP_SUFFIXES)

def psr_name(path): return os.path.basename(path).split('_')[0]

need_build = not os.path.isfile(BUNDLE)
parfiles, timfiles, common = [], [], []

if not need_build:
    print(f'✓ {BUNDLE} found — Cells 1 and 2 have nothing to do.')
    print('  Skip ahead to Cell 3.')
else:
    if not os.path.isfile('pta_paths.sh'):
        raise FileNotFoundError(
            'Neither the light data bundle nor pta_paths.sh was found.\n'
            'Either copy Student_data/ng15yr_light_data.pkl into place, or\n'
            'run  bash get_data.sh  in a terminal and re-run this cell.'
        )

    paths = load_pta_paths('pta_paths.sh')

    EXPECTED = ['NG15_PAR_DIR', 'NG15_TIM_DIR']
    missing  = [k for k in EXPECTED if k not in paths]
    if missing:
        raise KeyError(
            f'pta_paths.sh is missing: {missing}\n'
            'Re-run  bash get_data.sh  — it rewrites pta_paths.sh in ~2 s.'
        )

    pardir = paths['NG15_PAR_DIR']
    timdir = paths['NG15_TIM_DIR']

    # ── OVERRIDE BLOCK — edit if you moved the data ──────────────────────
    # pardir = '/my/custom/path/narrowband/par'
    # timdir = '/my/custom/path/narrowband/tim'
    # ─────────────────────────────────────────────────────────────────────

    parfiles_all = sorted(glob.glob(os.path.join(pardir, '*.par')))
    timfiles_all = sorted(glob.glob(os.path.join(timdir, '*.tim')))
    par_map = {psr_name(p): p for p in parfiles_all if is_canonical(p)}
    tim_map = {psr_name(t): t for t in timfiles_all if is_canonical(t)}
    matched = sorted(set(par_map) & set(tim_map))

    subsets = [n for n in matched
               if any(n.endswith(s) and n[:-len(s)] in matched for s in TEL_SUFFIXES)]
    common   = [n for n in matched if n not in subsets]
    parfiles = [par_map[n] for n in common]
    timfiles = [tim_map[n] for n in common]

    print(f'pardir : {pardir}')
    print(f'timdir : {timdir}')
    print(f'Total par files   : {len(parfiles_all)}')
    print(f'Matched par/tim   : {len(matched)}')
    print(f'Single-telescope subsets dropped : {len(subsets)}  {" ".join(subsets)}')
    print(f'Pulsars to build  : {len(parfiles)}  (expected: 68)')
    if len(parfiles) < 60:
        print('WARNING: fewer than 60 pulsars. Check pardir/timdir paths.')

In [ ]:
# ── Cell 2: Build the light data bundle from par/tim ────────────────────
# Runs only if Student_data/ng15yr_light_data.pkl is missing.  Takes ~25 min
# the first time; every pulsar is cached to Student_data/_cache/ as it is
# finished, so interrupting and re-running is safe.
#
# What we keep per pulsar:  name, sky position, TOA span, and the strain
# power sensitivity S_I(f).  That is all the sensitivity-curve machinery in
# hasasia ever reads, so the bundle is ~0.3 MB instead of ~100 GB.
#
# TOAs are thinned by 10 (as in the hasasia real-data tutorial): NANOGrav
# keeps >50 TOAs per observing epoch and hasasia inverts an N_TOA x N_TOA
# matrix, which is not tractable unthinned.  The floor of 5 rows per timing
# parameter keeps the design matrix from going degenerate on sparse pulsars.

EPHEM, THIN, MAX_TOAS, ROWS_PER_PARAM = 'DE438', 10, 6000, 5
TSPAN_YR, NFREQ, FMAX = 16.03, 400, 2e-7

freqs = np.logspace(np.log10(1.0 / (5 * TSPAN_YR * 365.25 * 24 * 3600)),
                    np.log10(FMAX), NFREQ)

def extract_one(name, parfile, timfile):
    """PINT -> thinned TOAs, errors, design matrix, sky position, noise basis."""
    cpath = os.path.join(CACHEDIR, f'{name}.npz')
    if os.path.exists(cpath):
        with np.load(cpath) as z:
            return {k: z[k] for k in z.files}, True

    import astropy.units as u
    from pint.models import get_model
    from pint.toa import get_TOAs

    model = get_model(parfile)
    toas  = get_TOAs(timfile, model=model, ephem=EPHEM, usepickle=True,
                     picklefilename=os.path.join(CACHEDIR, f'{name}.toas.pickle.gz'))

    n    = toas.ntoas
    t_s  = np.asarray(toas.get_mjds().to(u.s).value, dtype=float)
    M, _, _ = model.designmatrix(toas)
    M    = np.asarray(M, dtype=float)

    # EFAC / EQUAD, if PINT exposes the scaled uncertainties
    try:
        sigma = np.asarray(model.scaled_toa_uncertainty(toas).to(u.s).value, dtype=float)
    except Exception:
        sigma = np.asarray(toas.get_errors().to(u.s).value, dtype=float)

    # ECORR + power-law red noise, as a basis F with weights w: C = F diag(w) F^T
    try:
        F = np.asarray(model.noise_model_designmatrix(toas), dtype=float)
        w = np.asarray(model.noise_model_basis_weight(toas), dtype=float)
        if F.ndim != 2 or F.shape[1] != w.size:
            F, w = None, None
    except Exception:
        F, w = None, None

    # Ecliptic longitude and co-latitude, which is what hasasia wants
    c  = model.get_psr_coords()
    ec = c.barycentricmeanecliptic
    phi, theta = float(ec.lon.rad), float(np.pi / 2 - ec.lat.rad)

    n_par = M.shape[1]
    thin  = max(THIN, int(np.ceil(n / MAX_TOAS)))
    thin  = max(1, min(thin, n // max(ROWS_PER_PARAM * n_par, 1)))
    idx   = np.arange(0, n, thin)
    if idx.size < ROWS_PER_PARAM * n_par:
        raise ValueError(f'{idx.size} TOAs for {n_par} timing parameters — too sparse')

    out = {'toas': t_s[idx], 'toaerrs': sigma[idx], 'M': M[idx, :],
           'phi': np.array(phi), 'theta': np.array(theta),
           'thin': np.array(thin), 'n_par': np.array(n_par),
           'ntoa_full': np.array(n)}
    if F is not None:
        out['F'], out['w'] = F[idx, :], w

    os.makedirs(CACHEDIR, exist_ok=True)
    np.savez_compressed(cpath, **out)
    return out, False

def strain_sensitivity(d, freqs):
    """hasasia Spectrum for one pulsar; return its strain power sensitivity S_I."""
    N = np.diag(d['toaerrs'] ** 2)
    if 'F' in d:
        N = N + (d['F'] * d['w']) @ d['F'].T
    # unit-norm columns leave the design matrix's column space (and so hasasia's
    # G matrix) unchanged, but the SVD behind it is far better conditioned
    M = d['M'].copy()
    nrm = np.linalg.norm(M, axis=0); nrm[nrm == 0] = 1.0
    psr = hsen.Pulsar(toas=d['toas'], toaerrs=d['toaerrs'],
                      phi=float(d['phi']), theta=float(d['theta']),
                      N=N, designmatrix=M / nrm)
    return np.asarray(hsen.Spectrum(psr, freqs=freqs).S_I, dtype=float)

if not need_build:
    print('Nothing to build.')
else:
    os.makedirs(CACHEDIR, exist_ok=True)
    rows = []
    for i, (nm, pf, tf) in enumerate(zip(common, parfiles, timfiles), 1):
        try:
            d, cached = extract_one(nm, pf, tf)
            S_I = strain_sensitivity(d, freqs)
            if not np.all(np.isfinite(S_I)) or np.any(S_I <= 0):
                print(f'  ✗ {nm}: sensitivity is not finite and positive — skipped')
                continue
            rows.append({'name': nm, 'S_I': S_I,
                         'phi': float(d['phi']), 'theta': float(d['theta']),
                         'toa_range': (float(d['toas'].min()), float(d['toas'].max()))})
            print(f'  ✓ [{i:3d}/{len(common)}] {nm:12s} '
                  f'{int(d["ntoa_full"]):6d} TOAs → {d["toas"].size:5d} '
                  f'(thin {int(d["thin"])}, {int(d["n_par"])} params)'
                  + ('  [cached]' if cached else ''))
        except Exception as e:
            print(f'  ✗ {nm}: {type(e).__name__}: {e}')

    db = {'names': [r['name'] for r in rows], 'freqs': freqs,
          'S_I': np.array([r['S_I'] for r in rows]),
          'phi': np.array([r['phi'] for r in rows]),
          'theta': np.array([r['theta'] for r in rows]),
          'toa_range': np.array([r['toa_range'] for r in rows])}
    os.makedirs('Student_data', exist_ok=True)
    with open(BUNDLE, 'wb') as fh:
        pickle.dump(db, fh)
    print(f'\n✓ Wrote {BUNDLE} — {len(rows)} pulsars, '
          f'{os.path.getsize(BUNDLE)/1e6:.2f} MB')

In [ ]:
# ── Cell 3: Load the light data bundle ──────────────────────────────────
with open(BUNDLE, 'rb') as fh:
    db = pickle.load(fh)

names     = db['names']
freqs     = db['freqs']
S_I       = db['S_I']          # (N_psr, N_freq) strain power sensitivity
phi       = db['phi']          # ecliptic longitude [rad]
theta     = db['theta']        # ecliptic co-latitude [rad]
toa_range = db['toa_range']    # (N_psr, 2) first and last TOA [s]
Npsr      = len(names)

class LiteSpectrum:
    """Stand-in for hasasia.Spectrum carrying only what it reads downstream.

    hasasia's sensitivity-curve classes only ever touch .phi, .theta, .freqs,
    .S_I and .toas — and from .toas only min() and max(), for the total and
    pairwise timespans.  Holding the two endpoints reproduces both exactly, so
    the curves below come out of hasasia's own code rather than a formula
    rewritten here.
    """
    def __init__(self, name, freqs, S_I, phi, theta, toa_min, toa_max):
        self.name, self.freqs, self.S_I = name, freqs, np.asarray(S_I, dtype=float)
        self.phi, self.theta = float(phi), float(theta)
        self.toas = np.array([float(toa_min), float(toa_max)])

spectra = [LiteSpectrum(n, freqs, s, p, t, a, b)
           for n, s, p, t, (a, b) in zip(names, S_I, phi, theta, toa_range)]

Tspan = hsen.get_Tspan(spectra)

print(f'✓ Loaded {Npsr} pulsars from {BUNDLE}')
print(f'Tspan = {Tspan/365.25/24/3600:.1f} yr')
print(f'Frequency range: {freqs[0]:.2e} — {freqs[-1]:.2e} Hz  ({len(freqs)} bins)')
print(f'Pulsar pairs: {Npsr*(Npsr-1)//2}')

### ❓ Worksheet Q — Section 1

**Q:** What does the rms level of the timing residuals tell you about the sensitivity
of this pulsar as a gravitational-wave detector?
> Hint: think about the minimum strain $h$ that could produce a detectable timing
> modulation $\delta t \sim h / (2 \pi f)$ comparable to $\sigma_{\rm ToA}$.

In [ ]:
# YOUR ANSWER
answer_S0 = '''

'''
print(answer_S0)

---
## Section 1 · `hasasia` Sensitivity Curves and Hellings-Downs ORF
*(Worksheet: Section 3 — 'Module 2')*

In [ ]:
# ── Cell 4: Individual pulsar sensitivity curves ────────────────────────
# h_c = sqrt(f * S_I) is the characteristic strain a single pulsar is sensitive
# to.  Plotting h_c rather than sqrt(S_I) keeps this panel on the same
# (dimensionless) axis as the background in Cell 5.

fig, ax = plt.subplots(figsize=(10, 6))
cmap   = plt.cm.viridis
n_show = min(8, Npsr)
for k in range(n_show):
    ax.loglog(freqs, np.sqrt(freqs * S_I[k]), lw=1.4,
              color=cmap(k / n_show), alpha=0.85, label=names[k])
for k in range(n_show, Npsr):
    ax.loglog(freqs, np.sqrt(freqs * S_I[k]), color='k', lw=0.2, alpha=0.3)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel(r'Characteristic strain $h_c = \sqrt{f\,S_I(f)}$')
ax.set_title(f'Individual pulsar sensitivity — first {n_show} of {Npsr} pulsars')
ax.set_xlim(freqs.min(), freqs.max())
ax.grid(which='both', alpha=0.25)
ax.legend(fontsize=7, ncol=2); plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 5: Full PTA sensitivity curve ──────────────────────────────────
# GWBSensitivityCurve combines the single-pulsar sensitivities over every pair,
# weighted by the Hellings-Downs coefficient and the pair's overlapping
# timespan.  That pair weighting is the whole point: it is what makes a PTA a
# detector rather than 68 independent clocks.

sc_gwb = hsen.GWBSensitivityCurve(spectra)

A_gw    = 2.4e-15   # NANOGrav 15-yr amplitude at f = 1/yr, gamma = 13/3
hc_sgwb = A_gw * (freqs / fyr)**(-2/3)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(freqs, sc_gwb.h_c, color='steelblue', lw=2.5,
          label=r'PTA sensitivity $h_c$ (Hellings-Downs)')
ax.loglog(freqs, hc_sgwb, color='tomato', lw=2, linestyle='--',
          label=r'SGWB $h_c\propto f^{-2/3}$ ($A=2.4\times10^{-15}$)')
ax.axvline(1.0 / Tspan, color='gray', lw=1, linestyle=':')
ax.text(1.08 / Tspan, 0.92, r'$1/T_{\rm span}$', color='gray', fontsize=9,
        transform=ax.get_xaxis_transform())
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel(r'Characteristic strain $h_c$')
ax.set_title('NANOGrav 15-yr PTA sensitivity vs SGWB (Agazie et al. 2023)')
ax.set_xlim(freqs.min(), freqs.max())
ax.grid(which='both', alpha=0.25)
ax.legend(); plt.tight_layout(); plt.show()

i_best = np.argmin(sc_gwb.h_c)
print(f'Most sensitive at f = {freqs[i_best]:.2e} Hz, h_c = {sc_gwb.h_c[i_best]:.2e}')
print(f'Background at that frequency : h_c = {hc_sgwb[i_best]:.2e}')

In [ ]:
# ── Cell 6: Hellings-Downs ORF ──────────────────────────────────────────
def hellings_downs(zeta_rad):
    """Analytic Hellings-Downs overlap reduction function."""
    x = (1.0 - np.cos(zeta_rad)) / 2.0
    x = np.where(x < 1e-10, 1e-10, x)
    return 0.5 - 0.25 * x + 1.5 * x * np.log(x)

# Pairwise angular separations and HD coefficients, straight from hasasia so
# the pairs here are in the same order as everywhere else in the lab.
zetas, chi_pairs, pairs, _ = hsen.HellingsDownsCoeff(phi, theta)
ii, jj = pairs

zeta_grid  = np.linspace(0.01, np.pi, 500)
hd_theory  = hellings_downs(zeta_grid)
chi_scalar = 3/8 + np.cos(zeta_grid) / 8     # scalar-transverse (breathing) mode

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(np.degrees(zeta_grid), hd_theory,
        color='steelblue', lw=2.5, label='Hellings-Downs (GR tensor)')
ax.plot(np.degrees(zeta_grid), chi_scalar,
        color='darkorange', lw=2, linestyle='-.', label='Scalar breathing mode')
ax.plot(np.degrees(zetas), chi_pairs, '.', color='gray', ms=4, alpha=0.5,
        label=f'{len(zetas)} NANOGrav pulsar pairs')
ax.axhline(0,   color='gray', lw=0.8, linestyle='--')
ax.axhline(0.5, color='gray', lw=0.8, linestyle=':')
ax.annotate(r'$\chi(0)=1/2$', xy=(5, 0.5), xytext=(20, 0.52), fontsize=9,
            color='gray', arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))
ax.set_xlabel(r'Angular separation $\zeta$ (degrees)')
ax.set_ylabel(r'$\chi(\zeta)$')
ax.set_title('Hellings-Downs ORF (GR tensor) vs scalar breathing mode')
ax.legend(fontsize=9); ax.set_xlim(0, 180); plt.tight_layout(); plt.show()

print(f'Number of pulsar pairs: {len(pairs[0])}')
print(f'Angular separations range: {np.degrees(zetas.min()):.1f}° – {np.degrees(zetas.max()):.1f}°')
print(f'Minimum of HD curve: χ = {hd_theory.min():.3f} at '
      f'ζ = {np.degrees(zeta_grid[np.argmin(hd_theory)]):.0f}°')

### ❓ Worksheet Qs — Section 1 (hasasia)

**Q (sensitivity curves):** Why do individual pulsar sensitivity curves have a
characteristic U-shape — rising at both low and high frequency?
Explain the two distinct noise regimes.

**Q (HD ORF):** Answer the following:
1. What is qualitatively different between the GR tensor (HD) and scalar breathing ORFs?
2. Why does $\chi(0) = 1/2$ for the GR case?
3. What does a negative $\chi$ around $120°$ mean physically?

In [ ]:
# YOUR ANSWERS
answer_sensitivity_shape = '''
Low-f regime: [your answer]
High-f regime: [your answer]
'''

answer_HD_vs_scalar = '''
1. [your answer]
2. [your answer]
3. [your answer]
'''

print(answer_sensitivity_shape)
print(answer_HD_vs_scalar)

---
## Section 2 · Optimal Statistic and SGWB Evidence
*(Worksheet: Section 4 — 'Module 3')*

In [ ]:
# ── Cell 7: Optimal cross-correlation statistic ─────────────────────────
# The optimal statistic estimates the background amplitude A^2 from the
# HD-weighted sum of all pulsar-pair cross-correlations (Anholm et al. 2009,
# Chamberlin et al. 2015):
#
#     rho_ab   = pair estimate of A^2,  with variance sigma_ab^2
#     1/sigma_ab^2 = 2 T_ab * integral [ (S_h/A^2)^2 / (S_a S_b) ] df
#     A^2_hat  = sum(rho chi / sigma^2) / sum(chi^2 / sigma^2)
#     S/N      = sum(rho chi / sigma^2) / sqrt(sum(chi^2 / sigma^2))
#
# IMPORTANT — the rho_ab below are DRAWN, not measured. The real optimal
# statistic needs the timing residuals themselves, which the light data bundle
# deliberately throws away. What is real here is the geometry (chi_ab from the
# actual NANOGrav sky positions), the per-pulsar sensitivities S_a, and the
# pair timespans. So this is a realisation of what the 15-yr array would see
# given a background of amplitude A_gw, not a measurement of one.
#
# S_a includes the background itself as a noise term: once the GWB is loud
# enough to be seen it is also the dominant red noise in every pulsar, and
# leaving that out inflates the S/N by two orders of magnitude.

N_MODES = 14        # NANOGrav fits 14 Fourier modes, f = k/T_span
rng_os  = np.random.default_rng(1234)

band = (freqs >= 1.0 / Tspan) & (freqs <= N_MODES / Tspan)
fb   = freqs[band]
Sh   = (A_gw * (fb / fyr)**(-2/3))**2 / fb      # background strain power
S_tot = S_I[:, band] + Sh[None, :]              # noise + GWB self-noise

T_ab = (np.minimum(toa_range[ii, 1], toa_range[jj, 1])
        - np.maximum(toa_range[ii, 0], toa_range[jj, 0]))

inv_var = 2 * T_ab * np.trapz((Sh / A_gw**2)**2 / (S_tot[ii] * S_tot[jj]), fb, axis=1)
sig_rho = 1.0 / np.sqrt(inv_var)

# one realisation of the pair correlations
rho = A_gw**2 * chi_pairs + sig_rho * rng_os.standard_normal(chi_pairs.size)

wgt    = 1.0 / sig_rho**2
OS_val = np.sum(rho * chi_pairs * wgt) / np.sum(chi_pairs**2 * wgt)
OS_sig = 1.0 / np.sqrt(np.sum(chi_pairs**2 * wgt))
snr    = OS_val / OS_sig

print(f'Frequencies used   : {band.sum()} bins, '
      f'{fb[0]:.2e} — {fb[-1]:.2e} Hz')
print(f'Injected           A² = {A_gw**2:.3e}   (A = {A_gw:.2e})')
print(f'Optimal Statistic  A² = {OS_val:.3e} ± {OS_sig:.3e}')
print(f'Recovered          A  = {np.sqrt(max(OS_val, 0)):.3e}')
print(f'S/N                   = {snr:.2f}')

# Bin by angular separation
nbins    = 9
zeta_deg = np.degrees(zetas)
edges    = np.linspace(0, 180, nbins + 1)
rho_bin, _, _ = binned_statistic(zeta_deg, rho, statistic='mean', bins=edges)
var_bin, _, _ = binned_statistic(zeta_deg, sig_rho**2, statistic='mean', bins=edges)
counts, _     = np.histogram(zeta_deg, bins=edges)
zeta_mid      = 0.5 * (edges[:-1] + edges[1:])
rho_err       = np.sqrt(var_bin / np.where(counts > 0, counts, 1))

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(zeta_deg, rho, yerr=sig_rho,
            fmt='.', color='gray', alpha=0.35, ms=3, label='All pairs')
ax.errorbar(zeta_mid, rho_bin, yerr=rho_err,
            fmt='o', color='steelblue', ms=6, capsize=3, label='Binned')
ax.plot(np.degrees(zeta_grid), OS_val * hellings_downs(zeta_grid),
        color='tomato', lw=2, label=r'HD $\times\,\hat{A}^2$')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('Angular separation (degrees)')
ax.set_ylabel(r'Cross-correlation $\hat{\rho}$')
ax.set_title(f'Optimal Statistic S/N = {snr:.2f}')
ax.legend(fontsize=9); ax.set_xlim(0, 180); plt.tight_layout(); plt.show()

### ❓ Worksheet Qs — Section 2 (Optimal Statistic)

*Note: the pair correlations in Cell 7 are a simulated realisation built on the
real pulsar geometry and sensitivities — read the header comment before you
answer Q1.*

**Q1.** What S/N did you measure? Compare to the published 15-yr result (~4σ,
Agazie et al. 2023). What factors drive the S/N?

**Q2.** Qualitatively assess the agreement between the binned cross-correlations
and the HD curve. Do the data support the HD pattern?
What systematic effects could mimic or mask the HD signal?

**Q3.** If you doubled the number of pulsars while keeping $T_{\rm span}$ fixed,
how much would S/N improve? Show your reasoning using
S/N $\propto \sqrt{N_{\rm psr}(N_{\rm psr}-1)/2}$.

In [ ]:
# YOUR ANSWERS
measured_snr = None   # <-- replace with your number from Cell 7

answer_Q1 = f'''
Q1: S/N = {measured_snr}
    [your comparison and reasoning]
'''

answer_Q2 = '''
Q2: [your assessment]
    Possible systematics: [your answer]
'''

answer_Q3 = '''
Q3: [show your working]
'''

print(answer_Q1); print(answer_Q2); print(answer_Q3)

---
## Section 3 · `holodeck` Population Constraints
*(Worksheet: Section 5 — 'Module 4')*

In [ ]:
# ── Cell 8: holodeck — SMBHB GWB population simulation ──────────────────
# holodeck builds a semi-analytic model (SAM) of the cosmological SMBHB
# population and generates realisations of the GW background spectrum.
# Key inputs:
#   GSMF  — galaxy stellar mass function (Schechter)
#   GPF   — galaxy pair fraction
#   GMT   — galaxy merger timescale
#   MMBulge — M_BH–M_bulge scaling relation (Kormendy & Ho 2013)
#   Hardening — GW-driven, circular orbits
#
# sam.gwb() takes frequency bin EDGES and returns (hc_ss, hc_bg): the loudest
# single sources and the characteristic strain of the background, the latter
# with shape (N_freq, N_realisations).  SAM_SHAPE sets the (M, q, z) grid; the
# holodeck default is finer and about 20x slower, which is too slow for a lab.

SAM_SHAPE = (41, 41, 51)
N_REAL    = 100

sam_model = sams.Semi_Analytic_Model(
    gsmf=sams.GSMF_Schechter(),
    gpf=sams.GPF_Power_Law(),
    gmt=sams.GMT_Power_Law(),
    mmbulge=host_relations.MMBulge_KH2013(),
    shape=SAM_SHAPE,
)
hard = hardening.Hard_GW()

f_edges    = np.logspace(-9, -7, 41)                      # 1–100 nHz
freqs_holo = np.sqrt(f_edges[:-1] * f_edges[1:])          # bin centres
print(f'Generating {N_REAL} GWB realisations (may take ~30-60 s) ...')
hc_ss, hc_bg = sam_model.gwb(f_edges, hard=hard, realize=N_REAL)

hc_median = np.median(hc_bg, axis=1)
hc_lo     = np.percentile(hc_bg,  5, axis=1)
hc_hi     = np.percentile(hc_bg, 95, axis=1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(freqs_holo, hc_median, color='steelblue', lw=2.5,
          label=f'Median of {N_REAL} realisations')
ax.fill_between(freqs_holo, hc_lo, hc_hi, alpha=0.25, color='steelblue',
                label='5–95% prediction band')
f_ref = np.logspace(-9, -7, 100)
ax.loglog(f_ref, A_gw*(f_ref/fyr)**(-2/3), color='tomato', lw=2,
          linestyle='--', label=r'NANOGrav 15-yr ($A=2.4\times10^{-15}$)')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel(r'Characteristic strain $h_c$')
ax.set_title('holodeck: SMBHB GWB population realisations')
ax.set_xlim(1e-9, 1e-7); ax.grid(which='both', alpha=0.25)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

fyr_idx = np.argmin(np.abs(freqs_holo - fyr))
print(f'At f = {freqs_holo[fyr_idx]:.2e} Hz (nearest bin to 1/yr):')
print(f'  holodeck median h_c = {hc_median[fyr_idx]:.2e}')
print(f'  5–95% range         = {hc_lo[fyr_idx]:.2e} — {hc_hi[fyr_idx]:.2e}')
print(f'  NANOGrav 15-yr      = {A_gw*(freqs_holo[fyr_idx]/fyr)**(-2/3):.2e}')

In [ ]:
# ── Cell 9: Constrain M_BH–M_bulge normalisation from A_gw ──────────────
# mamp_log10 is log10(M_BH/Msun) at M_bulge = 1e11 Msun.  The KH2013 default is
# 8.69; we sweep around and above it to find what normalisation the measured
# background amplitude would require under this SAM.

mu_vals = [8.4, 8.8, 9.2, 9.6]
A_vals  = []

for mu in mu_vals:
    print(f'  Running mu = {mu} ...', end='', flush=True)
    sam_mu = sams.Semi_Analytic_Model(
        mmbulge=host_relations.MMBulge_KH2013(mamp_log10=mu),
        gsmf=sams.GSMF_Schechter(),
        gpf=sams.GPF_Power_Law(),
        gmt=sams.GMT_Power_Law(),
        shape=SAM_SHAPE,
    )
    _, hc_mu = sam_mu.gwb(f_edges, hard=hardening.Hard_GW(), realize=30)
    A_vals.append(np.median(hc_mu[fyr_idx, :]))
    print(f' A_gw = {A_vals[-1]:.2e}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mu_vals, np.array(A_vals)*1e15, 'o-', color='steelblue', ms=8, lw=2)
ax.axhline(2.4, color='tomato', lw=2, linestyle='--',
           label=r'NANOGrav 15-yr: $A_{\rm gw}=2.4\times10^{-15}$')
ax.fill_between(mu_vals, 2.4*0.7, 2.4*1.3, alpha=0.15, color='tomato',
                label=r'$\pm30\%$ uncertainty')
ax.axvline(8.69, color='gray', lw=1, linestyle=':', label='KH2013 default (8.69)')
ax.set_xlabel(r'$\mu=\log_{10}(M_{\rm BH}/M_\odot)$ at $M_{\rm bulge}=10^{11}M_\odot$')
ax.set_ylabel(r'$A_{\rm gw}\times10^{15}$')
ax.set_title(r'Constraining $M_{\rm BH}$–$M_{\rm bulge}$ from SGWB amplitude')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

A_arr = np.array(A_vals)*1e15
mu_pref = None
for i in range(len(mu_vals)-1):
    if (A_arr[i]-2.4)*(A_arr[i+1]-2.4) < 0:
        mu_pref = (mu_vals[i] + (2.4-A_arr[i])/(A_arr[i+1]-A_arr[i])
                   *(mu_vals[i+1]-mu_vals[i]))
        print(f'Preferred mu ~ {mu_pref:.2f}  (KH2013 default is 8.69)')
if mu_pref is None:
    print('No crossing inside the sweep — widen mu_vals.')

### ❓ Worksheet Qs — Section 3 (holodeck)

**Q1.** Does the default SAM bracket the NANOGrav 15-yr measurement?
Does the measurement lie inside the 5–95% band?

**Q2.** What value of $\mu$ is preferred by the NANOGrav measurement?
What does this imply about SMBHB masses compared to nearby galaxies?

**Q3 (extension).** A hardening model with stellar loss-cone hardening would
predict a steeper spectral slope at $f \lesssim f_{\rm yr}$.
Sketch what such a spectrum looks like, and explain what PTA observable
could distinguish it from the GW-only hardening case.

In [ ]:
# YOUR ANSWERS
answer_Q1 = '''
Q1: [your answer — does the measurement fall inside the 5-95% band?]
'''
answer_Q2 = '''
Q2: mu_preferred ~ [read from Cell 9 output]
    Implication: [your answer]
'''
answer_Q3 = '''
Q3 (extension): [your answer]
'''
print(answer_Q1); print(answer_Q2); print(answer_Q3)

---
## Section 4 · Synthesis and Open Questions
*(Worksheet: 'Synthesis: Big Ideas and Open Questions')*

In [ ]:
# ── Cell 10: Summary — S/N scaling with N_psr ───────────────────────────
N_psr_vals = np.array([10, 20, 45, 68, 100, 200])
snr_scale  = np.sqrt(N_psr_vals * (N_psr_vals - 1) / 2)
snr_norm   = snr_scale / snr_scale[np.argmin(np.abs(N_psr_vals - 45))]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(N_psr_vals, snr_norm, 'o-', color='steelblue', lw=2, ms=7)
ax.axvline(45, color='silver',     lw=1.5, linestyle=':', label='NANOGrav 11-yr (45)')
ax.axvline(68, color='tomato',     lw=2,   linestyle='--', label='NANOGrav 15-yr (68)')
for n, s in zip(N_psr_vals, snr_norm):
    ax.annotate(f'{s:.1f}x', (n, s), textcoords='offset points',
                xytext=(4, 4), fontsize=8)
ax.set_xlabel(r'Number of pulsars $N_{\rm psr}$')
ax.set_ylabel('S/N (relative to 45 pulsars)')
ax.set_title(r'OS S/N scaling: $\propto\sqrt{N_{\rm psr}(N_{\rm psr}-1)/2}$')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

print('Full analysis chain summary:')
print(f"{'Module':<22} {'Tool':<15} {'Output'}")
print('-'*60)
rows = [
    ('Sensitivity curves', 'hasasia', 'S_I(f), h_c(f)'),
    ('HD ORF',             'hasasia', 'chi(zeta), pair separations'),
    ('Optimal statistic',  'hasasia', 'A_gw^2/sigma, S/N'),
    ('Population model',   'holodeck','M_BH-host constraints'),
]
for m, t, o in rows:
    print(f'{m:<22} {t:<15} {o}')
print('-'*60)
print(f'This run: {Npsr} pulsars, {Npsr*(Npsr-1)//2} pairs, '
      f'T = {Tspan/365.25/24/3600:.1f} yr, OS S/N = {snr:.2f}')

### ❓ Worksheet Qs — Synthesis

**Open questions for further study** (list at least 2):

1. Is the background truly astrophysical (SMBHBs) or cosmological?
   What spectral measurement could distinguish them?

2. [Add your own open question from today's discussion]

---
**Self-assessment checklist** (mark ✓ or ✗):

| Learning outcome | ✓ / ✗ |
|:---|:---:|
| Load and inspect NANOGrav 15-yr par/tim data via PINT | |
| Build hasasia Pulsar objects and compute sensitivity curves | |
| Explain the U-shape of pulsar sensitivity curves | |
| Compute and interpret the Hellings-Downs ORF | |
| Understand OS S/N and its scaling with $N_{\rm psr}$ and $T$ | |
| Qualitatively interpret a holodeck GWB population prediction | |

In [ ]:
# YOUR ANSWERS — open questions
open_question_1 = '''
[Your first open question and thoughts]
'''
open_question_2 = '''
[Your second open question and thoughts]
'''
print(open_question_1); print(open_question_2)